# Stats

In [1]:
import os
import pandas as pd

## Solution Dataset

In [2]:
solutions_dataset = pd.read_csv('datasets/selected_problems_python_14B_no_comments_final.csv')
solutions_dataset.head()

,title,description,difficulty,source,solution,solution_no_comments
0,Sports Rank,Sports leagues rank their participants (teams ...,Medium,Kattis,def read_input():\n import sys\n input =...,def read_input():\n import sys\n input =...
1,Solidus Fraction,A fraction written with a solidus (/) can be w...,Medium,Kattis,"def gcd(a, b):\n while b != 0:\n a, ...","def gcd(a, b):\n while b != 0:\n a, ..."
2,Breakout,A variant of the popular video game breakout u...,Medium,Kattis,"def simulate_brick_wall(b, w, h, balls):\n ...","def simulate_brick_wall(b, w, h, balls):\n ..."
3,Passport Dates,Konráð has managed to obtain a big pile of Ice...,Easy,Kattis,def convert_date_format(input_date):\n # Sp...,def convert_date_format(input_date):\n part...
4,Weather Forecast,One of the most important things you can do in...,Hard,Kattis,def parse_input():\n import sys\n input ...,def parse_input():\n import sys\n input ...


In [3]:
solutions_dataset['line_count'] = solutions_dataset['solution_no_comments'].apply(lambda s: s.count('\n') + 1 if pd.notnull(s) else 0)
avg_lines_by_difficulty = solutions_dataset.groupby('difficulty')['line_count'].mean().reset_index()
avg_lines_by_difficulty = avg_lines_by_difficulty.sort_values(by='line_count', ascending=False)
avg_lines_by_difficulty

,difficulty,line_count
1,Hard,50.0
2,Medium,40.2
0,Easy,25.8


## Ground Truth Dataset

In [4]:
ground_truth_dataset = pd.read_csv('datasets/evaluation_python_solutions_1.csv')
ground_truth_dataset = ground_truth_dataset[['problem_title', 'code_line', 'good']]

In [5]:
ground_truth_dataset['good'].value_counts(normalize=True) * 100

good
1    82.672414
0    17.327586
Name: proportion, dtype: float64

In [6]:
merged_gt_dataset = ground_truth_dataset.merge(solutions_dataset[['title', 'difficulty']], left_on='problem_title', right_on='title', how='left')
merged_gt_dataset.head()

,problem_title,code_line,good,title,difficulty
0,Sports Rank,def read_input():,1,Sports Rank,Medium
1,Sports Rank,import sys,1,Sports Rank,Medium
2,Sports Rank,input = sys.stdin.read,0,Sports Rank,Medium
3,Sports Rank,data = input().split(),0,Sports Rank,Medium
4,Sports Rank,index = 0,1,Sports Rank,Medium


In [7]:
merged_gt_dataset.groupby('difficulty')['good'].mean() * 100

difficulty
Easy      82.558140
Hard      80.600000
Medium    85.323383
Name: good, dtype: float64

### Linter Crafted Dataset

In [8]:
crafted_dataset = pd.read_csv('datasets/all_lines_rf_dataset_no_sim.csv')
crafted_dataset.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,RUF059,SIM110,W291,W292,class,for,function,if,while,characters_number
0,A+BAgain?,1,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,14
1,A+BAgain?,2,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,15
2,A+BAgain?,3,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,14
3,A+BAgain?,4,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,16
4,A+BAgain?,5,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,16


In [9]:
crafted_dataset.columns

Index(['title', 'line_row', 'any_fix_available', 'all_fixes_available',
       'fix_availability_ratio', 'unsafe_fixes', 'safe_fixes',
       'avg_fix_complexity', 'max_fix_complexity', 'min_fix_complexity',
       'sum_fix_complexity', 'unique_issue_types', 'issues_per_line', 'A001',
       'ANN001', 'ANN202', 'ANN204', 'ARG001', 'B007', 'B905', 'C901',
       'COM812', 'D101', 'D102', 'D107', 'E226', 'E271', 'E301', 'E302',
       'E305', 'E306', 'E402', 'E501', 'E741', 'F821', 'F841', 'I001', 'N803',
       'N806', 'PERF401', 'PLC0415', 'PLC1901', 'PLR0913', 'PLR0914',
       'PLR0917', 'PLR1730', 'PLR2004', 'PLR6104', 'PLW0603', 'PLW2901',
       'Q000', 'RET504', 'RET505', 'RET507', 'RET508', 'RUF059', 'SIM110',
       'W291', 'W292', 'class', 'for', 'function', 'if', 'while',
       'characters_number'],
      dtype='object')

In [14]:
solutions_dataset['title_clean'] = solutions_dataset['title'].str.replace(' ', '', regex=False)

In [17]:
merged_crafted_dataset = crafted_dataset.merge(solutions_dataset[['title_clean', 'difficulty']], left_on='title', right_on='title_clean', how='left')

In [22]:
crafted_dataset['any_fix_available'].value_counts(normalize=True) * 100

any_fix_available
False    83.87931
True     16.12069
Name: proportion, dtype: float64

In [21]:
merged_crafted_dataset.groupby('difficulty')['any_fix_available'].value_counts(normalize=True) * 100

difficulty  any_fix_available
Easy        False                83.333333
            True                 16.666667
Hard        False                82.800000
            True                 17.200000
Medium      False                85.572139
            True                 14.427861
Name: proportion, dtype: float64

In [23]:
merged_crafted_dataset['all_fixes_available'].value_counts(normalize=True) * 100

all_fixes_available
False    88.017241
True     11.982759
Name: proportion, dtype: float64

In [24]:
merged_crafted_dataset.groupby('difficulty')['all_fixes_available'].value_counts(normalize=True) * 100

difficulty  all_fixes_available
Easy        False                  83.720930
            True                   16.279070
Hard        False                  89.400000
            True                   10.600000
Medium      False                  89.054726
            True                   10.945274
Name: proportion, dtype: float64

In [27]:
selected_columns = ['class', 'for', 'function', 'if', 'while']
print('Lines in a certain block:')
for column in selected_columns:
    print(f'{column}: {merged_crafted_dataset[column].sum()}')

Lines in a certain block:
class: 101
for: 509
function: 957
if: 317
while: 185


In [47]:
merged_crafted_dataset.groupby('difficulty')['issues_per_line'].sum()

difficulty
Easy      103
Hard      225
Medium    139
Name: issues_per_line, dtype: int64

In [40]:
merged_crafted_dataset['sum_fix_complexity'].max()

9.0

In [44]:
issue_columns = ['A001',
       'ANN001', 'ANN202', 'ANN204', 'ARG001', 'B007', 'B905', 'C901',
       'COM812', 'D101', 'D102', 'D107', 'E226', 'E271', 'E301', 'E302',
       'E305', 'E306', 'E402', 'E501', 'E741', 'F821', 'F841', 'I001', 'N803',
       'N806', 'PERF401', 'PLC0415', 'PLC1901', 'PLR0913', 'PLR0914',
       'PLR0917', 'PLR1730', 'PLR2004', 'PLR6104', 'PLW0603', 'PLW2901',
       'Q000', 'RET504', 'RET505', 'RET507', 'RET508', 'RUF059', 'SIM110',
       'W291', 'W292']
issues = {}
print('Lines with a certain issue:')
for column in issue_columns:
    issues[column] = merged_crafted_dataset[column].sum()
    # print(f'{column}: {merged_crafted_dataset[column].sum()}')
    
dict(sorted(issues.items(), key=lambda item: item[1], reverse=True))

Lines with a certain issue:


{'ANN001': 57,
 'E302': 37,
 'Q000': 31,
 'W292': 30,
 'E305': 27,
 'A001': 15.0,
 'PLC0415': 12,
 'W291': 10,
 'D102': 9,
 'E301': 9,
 'E501': 8,
 'RET504': 8,
 'N806': 7,
 'ANN204': 6,
 'D101': 6,
 'D107': 6,
 'E741': 6,
 'N803': 6,
 'E226': 5,
 'ANN202': 4,
 'ARG001': 4,
 'COM812': 4,
 'I001': 4,
 'B007': 3,
 'E402': 3,
 'F841': 3,
 'PERF401': 3,
 'PLR0913': 3,
 'PLR0917': 3,
 'PLR0914': 2,
 'PLR1730': 2,
 'PLW0603': 2,
 'RET505': 2,
 'RUF059': 2,
 'B905': 1,
 'C901': 1,
 'E271': 1,
 'E306': 1,
 'F821': 1,
 'PLC1901': 1,
 'PLR2004': 1,
 'PLR6104': 1,
 'PLW2901': 1,
 'RET507': 1,
 'RET508': 1,
 'SIM110': 1}